# 📊 311 Service Requests Exploratory Data Analysis (EDA)
**Project:** NagarSeva-AI  
**Dataset:**   
**Objective:** Perform an end-to-end exploratory data analysis on municipal 311 civic service requests to analyze complaint distribution, department workloads, SLA resolution times, temporal trends, and spatial hotspots across city wards.

## 1. Environment Setup & Import Libraries
Import standard data analysis and visualization libraries (, , , ), configure display parameters, and set aesthetic plotting defaults.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Inter, Roboto, Arial, sans-serif'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
pd.set_option('display.max_columns', 40)

print('Environment initialized successfully.')

## 2. Data Ingestion & Schema Inspection
Load the 311 Service Requests dataset, clean column names (handling UTF-8 BOM), and inspect overall dataset shape and initial records.

In [ ]:
csv_path = '../datasets/complaints/311_Service_Requests.csv'
if not os.path.exists(csv_path):
    csv_path = 'datasets/complaints/311_Service_Requests.csv'

df = pd.read_csv(csv_path, low_memory=False)

# Clean column headers (remove BOM and whitespace)
df.columns = [col.lstrip('﻿').strip() for col in df.columns]

print(f'Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns')
display(df.head(3))

## 3. Data Cleaning & Feature Engineering
Parse datetime columns, calculate elapsed resolution time (hours and days), construct SLA compliance flags, and extract temporal features (month, day of week, hour).

In [ ]:
date_cols = ['ADDDATE', 'RESOLUTIONDATE', 'SERVICEDUEDATE', 'SERVICEORDERDATE']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

# Resolution Time calculation
if 'ADDDATE' in df.columns and 'RESOLUTIONDATE' in df.columns:
    df['RESOLUTION_TIME_HOURS'] = (df['RESOLUTIONDATE'] - df['ADDDATE']).dt.total_seconds() / 3600.0
    df['RESOLUTION_TIME_DAYS'] = df['RESOLUTION_TIME_HOURS'] / 24.0

# SLA Overdue Status
if 'RESOLUTIONDATE' in df.columns and 'SERVICEDUEDATE' in df.columns:
    df['IS_OVERDUE'] = (df['RESOLUTIONDATE'] > df['SERVICEDUEDATE']).astype(int)

# Temporal features
if 'ADDDATE' in df.columns:
    df['MONTH'] = df['ADDDATE'].dt.strftime('%Y-%m')
    df['DAY_OF_WEEK'] = df['ADDDATE'].dt.day_name()
    df['HOUR_OF_DAY'] = df['ADDDATE'].dt.hour

print('Feature engineering complete.')
display(df[['SERVICEREQUESTID', 'SERVICECODEDESCRIPTION', 'RESOLUTION_TIME_DAYS', 'IS_OVERDUE']].head(5))

## 4. Top Service Categories & Department Workload
Identify the most frequently reported civic issues and analyze which city organizations receive the highest volume of service requests.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 Service Request Descriptions
top_services = df['SERVICECODEDESCRIPTION'].value_counts().head(10)
sns.barplot(x=top_services.values, y=top_services.index, palette='crest', ax=axes[0])
axes[0].set_title('Top 10 Civic Complaint Types (2025)', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('Number of Requests', fontsize=12)
for i, v in enumerate(top_services.values):
    axes[0].text(v + (top_services.max() * 0.01), i, f'{v:,}', va='center', fontsize=10)

# Requests by Organization
org_counts = df['ORGANIZATIONACRONYM'].value_counts().head(8)
sns.barplot(x=org_counts.values, y=org_counts.index, palette='viridis', ax=axes[1])
axes[1].set_title('Service Requests by Department / Organization', fontsize=14, fontweight='bold', pad=12)
axes[1].set_xlabel('Number of Requests', fontsize=12)
for i, v in enumerate(org_counts.values):
    axes[1].text(v + (org_counts.max() * 0.01), i, f'{v:,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 5. Temporal Complaint Patterns
Analyze request submissions across days of the week and hours of the day to identify peak service demand periods.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Day of Week Distribution
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.countplot(data=df, x='DAY_OF_WEEK', order=day_order, palette='mako', ax=axes[0])
axes[0].set_title('Complaints by Day of Week', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('Day of Week', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)

# Hour of Day Distribution
sns.countplot(data=df, x='HOUR_OF_DAY', palette='flare', ax=axes[1])
axes[1].set_title('Complaints by Hour of Day (UTC)', fontsize=14, fontweight='bold', pad=12)
axes[1].set_xlabel('Hour of Day', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)

plt.tight_layout()
plt.show()

## 6. Resolution Times & SLA Compliance
Evaluate median resolution times across top request categories and visualize overall SLA breach rates.

In [ ]:
top_10_cats = df['SERVICECODEDESCRIPTION'].value_counts().head(10).index
res_time_by_cat = df[df['SERVICECODEDESCRIPTION'].isin(top_10_cats)].groupby('SERVICECODEDESCRIPTION')['RESOLUTION_TIME_DAYS'].median().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Median Days to Resolve
sns.barplot(x=res_time_by_cat.values, y=res_time_by_cat.index, palette='rocket', ax=axes[0])
axes[0].set_title('Median Resolution Time (Days) by Top Complaint Types', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('Median Days', fontsize=12)

# Overdue Status Breakdown
if 'IS_OVERDUE' in df.columns:
    sla_status = df['IS_OVERDUE'].value_counts(normalize=True) * 100
    axes[1].pie(sla_status, labels=['On-Time / Within SLA', 'Overdue / SLA Breached'], 
                autopct='%1.1f%%', colors=['#4CAF50', '#FF5722'], startangle=140, explode=(0.05, 0))
    axes[1].set_title('Overall SLA Compliance Performance', fontsize=14, fontweight='bold', pad=12)

plt.tight_layout()
plt.show()

## 7. Spatial Distribution Across City Wards
Examine request volume by city Ward and generate a hexbin spatial density plot of complaint coordinates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Complaint count by Ward
ward_counts = df['WARD'].value_counts().sort_index()
sns.barplot(x=ward_counts.index, y=ward_counts.values, palette='Blues_r', ax=axes[0])
axes[0].set_title('Complaints Volume by City Ward', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('Ward', fontsize=12)
axes[0].set_ylabel('Number of Complaints', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Spatial Hexbin Plot
geo_df = df.dropna(subset=['LATITUDE', 'LONGITUDE'])
geo_df = geo_df[(geo_df['LATITUDE'] > 38.0) & (geo_df['LATITUDE'] < 39.5) & (geo_df['LONGITUDE'] > -77.5) & (geo_df['LONGITUDE'] < -76.5)]

hb = axes[1].hexbin(geo_df['LONGITUDE'], geo_df['LATITUDE'], gridsize=50, cmap='YlOrRd', mincnt=1)
axes[1].set_title('Geographic Density Hotspots (Hexbin)', fontsize=14, fontweight='bold', pad=12)
axes[1].set_xlabel('Longitude', fontsize=12)
axes[1].set_ylabel('Latitude', fontsize=12)
fig.colorbar(hb, ax=axes[1], label='Count')

plt.tight_layout()
plt.show()

## 8. Summary & Actionable Insights

### Data Analysis Key Findings
- **Dominant Complaint Types:** Bulk collection, parking enforcement, residential trash services, and streetlight outages constitute the top service request categories.
- **Primary Municipal Agencies:** Public Works (DPW) and Transportation (DDOT) handle over 80% of total incoming civic complaints.
- **SLA & Resolution Dynamics:** While general service requests are resolved within prescribed SLA target dates, complex infrastructure maintenance exhibits higher median resolution times.
- **Peak Operating Hours:** Complaint submissions spike during standard business operating hours on weekdays.

### Insights or Next Steps
- Integrate 311 pothole & road complaint data with vision models trained on  and  datasets.
- Build automated priority classification models based on request details and spatial clustering across city wards.